# Error de traslación de los loops — StereoLoopDetector en FieldSAFE

Para cada loop detectado (query `q`, match `m`) se compara la magnitud de la traslación estimada por PnP
(`loop_translation_x/y/z`, en el frame de la cámara) contra la distancia GPS planar entre ambos frames:

$$e = \left|\ \lVert t_{est} \rVert - \lVert p_{gps}[q] - p_{gps}[m] \rVert\ \right|$$

Se compara solo la magnitud porque el ground truth no tiene orientación y el detector descarta la rotación de PnP.

Corridas con `--frequency 10` (frecuencia nominal de la cámara) sobre las 3 sesiones de FieldSAFE:
dinámica #1, estática #1 y dinámica #2.

In [ ]:
from pathlib import Path
import numpy as np
import yaml
import matplotlib.pyplot as plt

BUILD = Path('../build')
PREPARED = Path('/mnt/datalake/datasets/fieldsafe/prepared')
FREQ = 10
SESSIONS = {  # nombre -> sesión de FieldSAFE
    'Dinámica #1': ('dynamic_1', '2016-10-25-11-41-21'),
    'Estática #1': ('static_1', '2016-10-25-11-09-42'),
    'Dinámica #2': ('dynamic_2', '2016-10-25-12-07-22'),
}
STOPPED_PATH_M = 1.0  # si el robot recorrió menos que esto entre match y query, el loop es "con robot detenido"

In [ ]:
def load_opencv_yml(path):
    # PyYAML no entiende la directiva %YAML:1.0 de OpenCV
    return yaml.safe_load(path.read_text().split('---', 1)[1])

def load_session(key, session):
    res = load_opencv_yml(BUILD / f'fieldSAFE_{key}_f{FREQ}_results.yml')
    poses = np.loadtxt(PREPARED / f'poses_{session}.csv', delimiter=',')
    assert len(poses) == res['num_images'] and (poses[:, 0] == np.arange(len(poses))).all()
    q = np.array(res['loop_query_ids'] or [], dtype=int)
    m = np.array(res['loop_match_ids'] or [], dtype=int)
    t_est = np.array([res['loop_translation_x'] or [], res['loop_translation_y'] or [],
                      res['loop_translation_z'] or []], dtype=float).T.reshape(-1, 3)
    return res, poses[:, 1:3], q, m, t_est

def translation_errors(xy, q, m, t_est):
    gt_dist = np.linalg.norm(xy[q] - xy[m], axis=1)
    err = np.abs(np.linalg.norm(t_est, axis=1) - gt_dist)
    cum_path = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(xy, axis=0), axis=1))])
    stopped = (cum_path[q] - cum_path[m]) < STOPPED_PATH_M
    return err, stopped

data = {}
for label, (key, session) in SESSIONS.items():
    res, xy, q, m, t_est = load_session(key, session)
    err, stopped = translation_errors(xy, q, m, t_est)
    data[label] = dict(res=res, err=err, stopped=stopped)
    print(f"{label}: {res['num_images']} imágenes, {len(q)} loops")

In [ ]:
def groups_of(d):
    e, s = d['err'], d['stopped']
    return [('Todos', e), ('En movimiento', e[~s]), ('Detenido', e[s])]

print(f"{'Sesión':<13} {'Grupo':<14} {'n':>4} {'media [m]':>10} {'mediana [m]':>12} {'p90 [m]':>8} {'max [m]':>8}")
for label, d in data.items():
    for name, e in groups_of(d):
        if len(e):
            print(f"{label:<13} {name:<14} {len(e):4d} {e.mean():10.3f} {np.median(e):12.3f} "
                  f"{np.percentile(e, 90):8.3f} {e.max():8.3f}")
        else:
            print(f"{label:<13} {name:<14} {0:4d} {'—':>10} {'—':>12} {'—':>8} {'—':>8}")

In [ ]:
box_color, ink, muted = '#3b6fb6', '#1f2328', '#6b7280'
Y_MAX = 1.5  # m; los puntos por encima se marcan con ▲ en el borde
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, len(data), figsize=(5 * len(data), 5.5), dpi=150, sharey=True)
for ax, (label, d) in zip(np.atleast_1d(axes), data.items()):
    groups = [(n, e) for n, e in groups_of(d) if len(e)]
    if groups:
        ax.boxplot([e for _, e in groups], widths=0.45, showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor='#dbe6f5', edgecolor=box_color, linewidth=1.5),
                   medianprops=dict(color=ink, linewidth=2),
                   whiskerprops=dict(color=box_color, linewidth=1.5),
                   capprops=dict(color=box_color, linewidth=1.5))
        for i, (_, e) in enumerate(groups, start=1):
            jitter = i + rng.uniform(-0.15, 0.15, len(e))
            inside = e <= Y_MAX
            ax.scatter(jitter[inside], e[inside], s=10, color=box_color, alpha=0.35, linewidths=0, zorder=3)
            if (~inside).any():
                ax.scatter(np.full((~inside).sum(), i), np.full((~inside).sum(), Y_MAX * 0.985), marker='^', s=30,
                           color=box_color, zorder=3, clip_on=False)
                ax.annotate(', '.join(f"{v:.2f}" for v in np.sort(e[~inside])) + ' m', (i + 0.08, Y_MAX * 0.985),
                            fontsize=7, color=muted, va='center')
            ax.scatter(i, e.mean(), marker='D', s=40, color='white', edgecolor=ink, linewidths=1.5, zorder=4)
            ax.annotate(f"media {e.mean():.3f}\nmediana {np.median(e):.3f}", (i, 1.01),
                        xycoords=('data', 'axes fraction'), ha='center', va='bottom', fontsize=7.5, color=ink,
                        annotation_clip=False)
        ax.set_xticks(range(1, len(groups) + 1), [f"{n}\n(n={len(e)})" for n, e in groups])
    else:
        ax.text(0.5, 0.5, 'sin loops detectados', transform=ax.transAxes, ha='center', color=muted)
        ax.set_xticks([])
    ax.set_ylim(-0.05, Y_MAX)
    ax.set_title(f"{label}  ({d['res']['num_images']} imágenes)", fontsize=10.5, color=ink, pad=32)
    ax.grid(axis='y', color='#e5e7eb', linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(muted)
    ax.tick_params(colors=muted)

np.atleast_1d(axes)[0].set_ylabel('Error de traslación |‖t_est‖ − ‖Δp_gps‖|  [m]', color=ink)
fig.suptitle(f"FieldSAFE — error de traslación por query (--frequency {FREQ})    "
             f"◆ media   — mediana   · cada query   ▲ fuera de escala (> {Y_MAX} m)", fontsize=11, color=ink)
fig.tight_layout()
fig.savefig('translation_error_boxplot.png')
plt.show()